# Working with Redis as a Vector Store

This notebook demonstrates the usage of the `RedisVectorStore` class from the langchain-redis package to enable efficient storage, retrieval, and similarity search of vector embeddings using Redis.

## Installation

In [ ]:
# %pip install ipywidgets
# %pip install langchain
# %pip install langchain-redis
# %pip install langchain-huggingface
# %pip install sentence-transformers

## Importing Required Libraries

In [1]:
import json
import redis

from redisvl.query.filter import Num
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_redis import RedisVectorStore

## Setting up Redis Connection

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

redis_url = os.getenv("REDIS_URL")
redis_client = redis.from_url(redis_url)
redis_client.ping()

True

## Preparing Sample Data

In [3]:
with open('data/movies.json', 'r') as file:
    data = json.load(file)
    documents = [
        Document(
            page_content=movie["info"]["plot"],
            metadata={
                "title": movie["title"],
                "year": movie["year"],
                "rating": movie["info"]["rating"],
            })
        for movie in data["movies"]
    ]

Let's inspect the first document:

In [4]:
documents[0]

Document(metadata={'title': 'Blade', 'year': 1998, 'rating': 7}, page_content='A half-vampire, half-mortal man becomes a protector of the mortal race, while slaying evil vampires.')

## Creating Embeddings

In [5]:
embeddings = HuggingFaceEmbeddings(model_name="msmarco-distilbert-base-v4")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/545 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/319 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## Using Redis with LangChain
Now let's play with Redis as a vector store using LangChain.

### Creating a vector store instance and inserting data
Use a batch or load in parallel for large file uploads as it will be very time consuming to load it in 1 shot

In [ ]:
vector_store = RedisVectorStore.from_documents(
    documents,
    embeddings,
    redis_url=redis_url,
    index_name="movies",
    metadata_schema=[
        {"name":"title", "type":"text"},
        {"name":"year", "type":"numeric"},
        {"name":"rating", "type":"numeric"},
    ],
    storage_type = "json"
)

### Performing a simple similarity search

In [9]:
query = "He seeks revenge for his family's death"
# k represents the number of results to return . close enough to the query,
results = vector_store.similarity_search(query, k=2)

for doc in results:
    print(doc)

page_content='After his wife and family are killed by criminals, FBI agent Frank Castle becomes a vigilante known as "The Punisher," who aims to fight crime by any means necessary.' metadata={'title': 'The Punisher', 'year': 2004, 'rating': 7.8}
page_content='After his wife and family are killed by criminals, FBI agent Frank Castle becomes a vigilante known as "The Punisher," who aims to fight crime by any means necessary.' metadata={'title': 'The Punisher', 'year': 2004, 'rating': 7.8}


### Similarity search with metadata filtering

In [10]:
filter_condition = Num("year") == 2004

filtered_results = vector_store.similarity_search(
    query, k=2, filter=filter_condition
)

for doc in filtered_results:
    print(doc)

page_content='After his wife and family are killed by criminals, FBI agent Frank Castle becomes a vigilante known as "The Punisher," who aims to fight crime by any means necessary.' metadata={'title': 'The Punisher', 'year': 2004, 'rating': 7.8}
page_content='After his wife and family are killed by criminals, FBI agent Frank Castle becomes a vigilante known as "The Punisher," who aims to fight crime by any means necessary.' metadata={'title': 'The Punisher', 'year': 2004, 'rating': 7.8}


### Similarity search with score

In [11]:
scored_results = vector_store.similarity_search_with_score(
    query, k=2, filter=filter_condition
)

for doc, score in scored_results:
    print(doc, score)

page_content='After his wife and family are killed by criminals, FBI agent Frank Castle becomes a vigilante known as "The Punisher," who aims to fight crime by any means necessary.' metadata={'title': 'The Punisher', 'year': 2004, 'rating': 7.8} 0.661215007305
page_content='After his wife and family are killed by criminals, FBI agent Frank Castle becomes a vigilante known as "The Punisher," who aims to fight crime by any means necessary.' metadata={'title': 'The Punisher', 'year': 2004, 'rating': 7.8} 0.661215007305


## Cleanup

In [12]:
# Delete the underlying index and it's data
vector_store.index.delete(drop=True)